# Age Group Estimation from Faces — CNN Image Classification

**Phase 2: Proposal & Technical Implementation**

This notebook implements a CNN-based classifier that estimates a person's **age group** from a facial image, using the **UTKFace** dataset.

**Dataset:** https://www.kaggle.com/datasets/jangedoo/utkface-new

The continuous age label (0–116) is binned into **6 age groups**, turning the task into a 6-class image classification problem.

| Class | Age range | Label |
|-------|-----------|-------|
| 0 | 0–12  | Child |
| 1 | 13–19 | Teenager |
| 2 | 20–34 | Young Adult |
| 3 | 35–49 | Adult |
| 4 | 50–64 | Middle-Aged |
| 5 | 65+   | Senior |

**Pipeline:** load & parse → EDA → preprocess (resize + normalize + augment) → stratified train/val/test split → custom CNN → MobileNetV2 transfer learning → evaluation (confusion matrix, classification report, accuracy/loss curves) → model comparison → Grad-CAM.

> **Note:** To download the MobileNetV2 ImageNet weights, enable **Internet** in the Kaggle notebook settings (Settings → Internet → On). A **GPU** accelerator is recommended for training.

In [ ]:
# ----------------------------- Imports & configuration -----------------------------
import os, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Configuration
IMG_SIZE   = (128, 128)          # all images resized to this
BATCH_SIZE = 64
N_CLASSES  = 6
CLASS_NAMES = ['0-12', '13-19', '20-34', '35-49', '50-64', '65+']
SAVE_DIR   = '/kaggle/working'   # figures are saved here
os.makedirs(SAVE_DIR, exist_ok=True)
saved_figures = []               # keep track of saved figure paths

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices('GPU')))

## 1. Dataset loading

UTKFace stores the labels **inside the filename**, formatted as
`[age]_[gender]_[race]_[datetime].jpg`. We only need the **age** (the first field).

A few files in the dataset have malformed names (a missing field), so the parser is
written defensively: any file whose age cannot be read, or that falls outside a valid
age range, is skipped.

In [ ]:
# Locate the image folder (the Kaggle dataset has a couple of possible layouts)
BASE = '/kaggle/input/utkface-new'
candidate_dirs = [
    os.path.join(BASE, 'UTKFace'),
    os.path.join(BASE, 'utkface_aligned_cropped', 'UTKFace'),
    os.path.join(BASE, 'crop_part1'),
    BASE,
]
image_dir = next((d for d in candidate_dirs if os.path.isdir(d) and glob.glob(os.path.join(d, '*.jpg'))), BASE)
print("Using image directory:", image_dir)

def age_to_group(age):
    # Map a continuous age to one of the 6 age-group labels
    if age <= 12:  return '0-12'
    if age <= 19:  return '13-19'
    if age <= 34:  return '20-34'
    if age <= 49:  return '35-49'
    if age <= 64:  return '50-64'
    return '65+'

# Parse every image into a (filepath, age, age_group) record
image_paths = glob.glob(os.path.join(image_dir, '*.jpg'))
records, skipped = [], 0
for path in image_paths:
    fname = os.path.basename(path)
    try:
        age = int(fname.split('_')[0])      # first field = age
    except (ValueError, IndexError):
        skipped += 1; continue              # malformed filename -> skip
    if 1 <= age <= 116:
        records.append({'filepath': path, 'age': age, 'age_group': age_to_group(age)})
    else:
        skipped += 1

df = pd.DataFrame(records)
print(f"Total images found : {len(image_paths)}")
print(f"Valid images parsed: {len(df)}")
print(f"Skipped (malformed): {skipped}")
df.head()

## 2. Exploratory data analysis

We inspect the **class distribution** (UTKFace is naturally imbalanced — most faces are
young adults) and look at a few sample images per age group.

In [ ]:
# Class distribution
dist = df['age_group'].value_counts().reindex(CLASS_NAMES)
print("Class distribution:")
print(dist)

plt.figure(figsize=(8, 4.5))
sns.barplot(x=dist.index, y=dist.values, palette='viridis')
plt.title('Age-group distribution (UTKFace)')
plt.xlabel('Age group'); plt.ylabel('Number of images')
for i, v in enumerate(dist.values):
    plt.text(i, v + 50, str(int(v)), ha='center', fontsize=9)
plt.tight_layout()
p = os.path.join(SAVE_DIR, 'class_distribution.png')
plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
plt.show()

In [ ]:
# Show one sample image per age group
from tensorflow.keras.utils import load_img
plt.figure(figsize=(14, 3))
for i, grp in enumerate(CLASS_NAMES):
    sample = df[df['age_group'] == grp].iloc[0]
    img = load_img(sample['filepath'], target_size=IMG_SIZE)
    plt.subplot(1, N_CLASSES, i + 1)
    plt.imshow(img); plt.axis('off')
    plt.title(f"{grp}\n(age {sample['age']})", fontsize=9)
plt.suptitle('Sample image per age group', y=1.05)
plt.tight_layout()
p = os.path.join(SAVE_DIR, 'sample_images.png')
plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
plt.show()

## 3. Train / validation / test split

A **stratified** 70 / 15 / 15 split keeps the age-group proportions identical across the
three sets, which matters because the dataset is imbalanced.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['age_group'], random_state=SEED)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['age_group'], random_state=SEED)

print(f"Train: {len(train_df):>6}  ({len(train_df)/len(df):.0%})")
print(f"Val  : {len(val_df):>6}  ({len(val_df)/len(df):.0%})")
print(f"Test : {len(test_df):>6}  ({len(test_df)/len(df):.0%})")

## 4. Preprocessing: resizing, normalization & data augmentation

`ImageDataGenerator` handles three of the required preprocessing steps in one place:
- **Resizing** every image to `128 x 128` (`target_size`).
- **Normalization** by rescaling pixels from `[0, 255]` to `[0, 1]` (`rescale=1./255`).
- **Augmentation** (training set only): small rotations, shifts, zoom and horizontal flips
  to reduce overfitting. Validation and test sets are only rescaled (never augmented).

Passing `classes=CLASS_NAMES` locks the label-to-index mapping so it is identical for the
train, validation and test generators — essential for a correct confusion matrix.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
)
eval_datagen = ImageDataGenerator(rescale=1./255)   # no augmentation for val/test

flow_args = dict(x_col='filepath', y_col='age_group', target_size=IMG_SIZE,
                 batch_size=BATCH_SIZE, class_mode='categorical',
                 classes=CLASS_NAMES, seed=SEED)

train_gen = train_datagen.flow_from_dataframe(train_df, shuffle=True,  **flow_args)
val_gen   = eval_datagen.flow_from_dataframe(val_df,   shuffle=False, **flow_args)
test_gen  = eval_datagen.flow_from_dataframe(test_df,  shuffle=False, **flow_args)

print("Class index mapping:", train_gen.class_indices)

In [ ]:
# Class weights to counter the dataset imbalance during training
labels = train_gen.classes
weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weight = {int(k): float(v) for k, v in zip(np.unique(labels), weights)}
print("Class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weight.items()})

## 5. Custom CNN model

A compact CNN built from four `Conv → BatchNorm → MaxPool` blocks (32 → 64 → 128 → 256
filters), followed by global average pooling and two dense layers with dropout. The model
is defined with the **Functional API** so its internal feature maps stay accessible for
Grad-CAM later. The final convolutional layer is named `last_conv` for that purpose.

In [ ]:
def build_custom_cnn(input_shape=(128, 128, 3), n_classes=N_CLASSES):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
    # final conv block (named for Grad-CAM)
    x = layers.Conv2D(256, 3, padding='same', activation='relu', name='last_conv')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    return models.Model(inputs, outputs, name='custom_cnn')

cnn = build_custom_cnn()
cnn.compile(optimizer=optimizers.Adam(1e-3),
            loss='categorical_crossentropy', metrics=['accuracy'])
cnn.summary()

In [ ]:
# Training callbacks: stop early on the best validation loss, and drop LR on plateau
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

EPOCHS_CNN = 30
history_cnn = cnn.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_CNN,
    class_weight=class_weight,
    callbacks=cb,
    verbose=1,
)

### 5.1 Accuracy and loss curves

In [ ]:
def plot_history(history, title, fname):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['accuracy'], label='train')
    ax[0].plot(h['val_accuracy'], label='val')
    ax[0].set_title(f'{title} — Accuracy'); ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Accuracy'); ax[0].legend()
    ax[1].plot(h['loss'], label='train')
    ax[1].plot(h['val_loss'], label='val')
    ax[1].set_title(f'{title} — Loss'); ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('Loss'); ax[1].legend()
    plt.tight_layout()
    p = os.path.join(SAVE_DIR, fname)
    plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
    plt.show()

plot_history(history_cnn, 'Custom CNN', 'cnn_curves.png')

### 5.2 Evaluation — test accuracy, confusion matrix & classification report

A single reusable function evaluates any model on the test set. Because the test generator
uses `shuffle=False`, the predicted labels line up exactly with `test_gen.classes`.

In [ ]:
def evaluate_model(model, gen, name, fname):
    # Predictions; gen has shuffle=False so order matches gen.classes
    probs = model.predict(gen, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_true = gen.classes

    test_acc = (y_pred == y_true).mean()
    print(f"\n{name} — Test accuracy: {test_acc:.4f}\n")
    print("Classification report:")
    print(classification_report(y_true, y_pred, labels=range(N_CLASSES),
                                target_names=CLASS_NAMES, zero_division=0))

    cm_mat = confusion_matrix(y_true, y_pred, labels=range(N_CLASSES))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm_mat, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{name} — Confusion matrix')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.tight_layout()
    p = os.path.join(SAVE_DIR, fname)
    plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
    plt.show()
    return test_acc, probs

cnn_acc, cnn_probs = evaluate_model(cnn, test_gen, 'Custom CNN', 'cnn_confusion_matrix.png')

## 6. Transfer learning: MobileNetV2

For comparison we fine-tune **MobileNetV2** (pre-trained on ImageNet). Training is done in
two stages:

1. **Feature extraction** — the convolutional base is frozen and only the new classification
   head is trained.
2. **Fine-tuning** — the top ~30 layers of the base are unfrozen and trained with a much
   lower learning rate so the pre-trained features adapt to faces without being destroyed.

> Requires Internet enabled in Kaggle to download the ImageNet weights.

In [ ]:
def build_mobilenet(input_shape=(128, 128, 3), n_classes=N_CLASSES):
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False                       # stage 1: freeze base
    inputs = layers.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = models.Model(inputs, outputs, name='mobilenetv2_transfer')
    return model, base

mnet, base = build_mobilenet()
mnet.compile(optimizer=optimizers.Adam(1e-3),
             loss='categorical_crossentropy', metrics=['accuracy'])
mnet.summary()

In [ ]:
# Stage 1 — train only the classification head
EPOCHS_HEAD = 10
history_tl_head = mnet.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=cb, verbose=1)

In [ ]:
# Stage 2 — unfreeze the top layers of the base and fine-tune at a low learning rate
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False     # keep early (generic) layers frozen

mnet.compile(optimizer=optimizers.Adam(1e-5),   # very low LR for fine-tuning
             loss='categorical_crossentropy', metrics=['accuracy'])

EPOCHS_FT = 10
history_tl_ft = mnet.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_FT, class_weight=class_weight, callbacks=cb, verbose=1)

In [ ]:
# Combine the two training stages into one history for plotting
class _H:
    def __init__(self, d): self.history = d
combined = {k: history_tl_head.history[k] + history_tl_ft.history[k]
            for k in history_tl_head.history}
plot_history(_H(combined), 'MobileNetV2 (transfer)', 'mobilenet_curves.png')

In [ ]:
mnet_acc, mnet_probs = evaluate_model(mnet, test_gen, 'MobileNetV2', 'mobilenet_confusion_matrix.png')

## 7. Model comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Custom CNN', 'MobileNetV2 (transfer)'],
    'Test accuracy': [round(cnn_acc, 4), round(mnet_acc, 4)],
    'Trainable params': [cnn.count_params(), mnet.count_params()],
})
print(comparison.to_string(index=False))

plt.figure(figsize=(6, 4))
sns.barplot(x='Model', y='Test accuracy', data=comparison, palette='Set2')
plt.ylim(0, 1); plt.title('Test accuracy comparison')
for i, v in enumerate(comparison['Test accuracy']):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center')
plt.tight_layout()
p = os.path.join(SAVE_DIR, 'model_comparison.png')
plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
plt.show()

## 8. Grad-CAM (model interpretability)

Grad-CAM highlights the regions of the face that most influenced the custom CNN's
prediction, by weighting the `last_conv` feature maps with the gradients of the predicted
class. Warm colours = more important regions.

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_name='last_conv'):
    grad_model = models.Model(model.inputs,
                              [model.get_layer(last_conv_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        class_idx = tf.argmax(preds[0])
        class_channel = preds[:, class_idx]
    grads = tape.gradient(class_channel, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.squeeze(conv_out @ pooled[..., tf.newaxis])
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(class_idx)

# Take one batch of test images and overlay Grad-CAM
imgs, labels = next(iter(test_gen))
n_show = 5
plt.figure(figsize=(3 * n_show, 6))
for i in range(n_show):
    img = imgs[i]
    heatmap, pred_idx = make_gradcam_heatmap(img[np.newaxis, ...], cnn)
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], IMG_SIZE).numpy().squeeze()
    true_idx = int(np.argmax(labels[i]))

    plt.subplot(2, n_show, i + 1)
    plt.imshow(img); plt.axis('off')
    plt.title(f"True: {CLASS_NAMES[true_idx]}\nPred: {CLASS_NAMES[pred_idx]}", fontsize=9)

    plt.subplot(2, n_show, n_show + i + 1)
    plt.imshow(img)
    plt.imshow(heatmap_resized, cmap='jet', alpha=0.5)
    plt.axis('off'); plt.title('Grad-CAM', fontsize=9)

plt.tight_layout()
p = os.path.join(SAVE_DIR, 'gradcam.png')
plt.savefig(p, dpi=120, bbox_inches='tight'); saved_figures.append(p)
plt.show()

## 9. Summary

The notebook covered the full required pipeline: dataset loading and robust filename
parsing, age-group binning, EDA, resizing + normalization + augmentation, a stratified
train/validation/test split with class weighting, a custom CNN, a MobileNetV2 transfer-
learning model trained in two stages, full evaluation (accuracy/loss curves, confusion
matrices, classification reports), a model comparison, and Grad-CAM interpretability.

All figures are saved to `/kaggle/working` and can be committed to the GitHub repository.

In [ ]:
print("Saved figures:")
for f in saved_figures:
    print(" -", f)